# 07 · Find a Tender (OCDS API)

Research log: [Step 8](../docs/research_log.md#step-8--achiziții-publice-find-a-tender-api-ocds) (in Romanian). Data and checks as of October 2026.

**Question.** Do UK public contract award notices contain the supplier's VAT number? Can they give an unbiased sample of companies with a website?

**Method.** Find a Tender OCDS release-package API (`stages=award`): identifier schemes, field completeness, then three sampling attempts; CH numbers checked against Companies House; domains checked.

**Result (from the research log).**
- 0 VAT identifiers in 1,480 supplier appearances; main identifier GB-COH for 49%.
- Framework agreements: 5% of contracts hold 61% of supplier appearances, so sampling must be per contract.
- 6,000 award notices Jan–Sep 2026: CPV 85 = 36%; only 1.3% of the other contracts have a supplier with CH number + website; CH numbers correct 30/30.

**Notes for re-running.**
- API rate limit: HTTP 429 with `Retry-After: 120` about every 10 pages.
- The attempt-2 cell overwrites `fat_esantion_30.csv`; the last cell overwrites `fat_esantion_30_v3.csv` with hard-coded manual domain labels.
- The 6,000-notice download goes to `../data/registre/fat_award_2026_01_09.jsonl` (not in git).

*Cells are kept as they were run, outputs included; only file paths were updated to the new `data/` layout and session timestamps in outputs were cut to the month.*

In [4]:
import time, json, collections
import requests

BAZA = "https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages"
HEADERS = {"User-Agent": "veridion-vat-poc (student research)"}
params = {"updatedFrom": "2026-09-01T00:00:00", "updatedTo": "2026-10-01T00:00:00",
          "limit": 100, "stages": "award"}

def cere(url, cerere, incercari=5):
    for i in range(incercari):
        r = requests.get(url, params=cerere, headers=HEADERS, timeout=60)
        if r.status_code in (429, 503):
            asteapta = int(r.headers.get("Retry-After", 10))
            print(f"  status {r.status_code}, aștept {asteapta}s (Retry-After)")
            time.sleep(asteapta)
            continue
        return r
    return r

scheme_principal = collections.Counter()
scheme_aditional = collections.Counter()
exemple_vat, exemple_fara = [], []
furnizori, releases, pagini = 0, 0, 0
url, cerere = BAZA, dict(params)

while url and pagini < 5:                      # max. 5 × 100 = 500 de release-uri
    r = cere(url, cerere)
    print("Pagina", pagini, "| status", r.status_code)
    if r.status_code != 200:
        print(r.text[:300]); break
    date = r.json()
    if pagini == 0:
        print("Chei în răspuns:", list(date.keys()), "| licență:", date.get("license"))
    for rel in date.get("releases", []):
        releases += 1
        for p in rel.get("parties", []):
            if "supplier" not in p.get("roles", []):
                continue
            furnizori += 1
            scheme_principal[p.get("identifier", {}).get("scheme", "LIPSĂ")] += 1
            for ai in p.get("additionalIdentifiers", []):
                scheme_aditional[ai.get("scheme", "LIPSĂ")] += 1
                if "VAT" in str(ai.get("scheme", "")).upper():
                    exemple_vat.append((p.get("name"), ai.get("id"), rel.get("ocid")))
            if len(exemple_fara) < 3:
                exemple_fara.append(p)
    nxt = date.get("links", {}).get("next")
    if nxt:
        url, cerere = nxt, None
    elif date.get("cursor") or date.get("nextCursor"):
        url, cerere = BAZA, {**params, "cursor": date.get("cursor") or date.get("nextCursor")}
    else:
        url = None
    pagini += 1
    time.sleep(1)

print(f"\nRelease-uri: {releases} | apariții de furnizori: {furnizori}")
print("Scheme principale:", dict(scheme_principal))
print("Scheme adiționale:", dict(scheme_aditional))
print("Furnizori cu TVA:", len(exemple_vat))
for e in exemple_vat[:10]:
    print("  ", e)
print("\nExemplu de furnizor (structura completă):")
print(json.dumps(exemple_fara[0] if exemple_fara else {}, indent=2)[:1500])

json.dump({"principal": scheme_principal, "aditional": scheme_aditional, "exemple_vat": exemple_vat},
          open("../data/registre/find_a_tender_scheme.json", "w"), indent=2, default=list)

Pagina 0 | status 200
Chei în răspuns: ['uri', 'version', 'extensions', 'publishedDate', 'publisher', 'license', 'publicationPolicy', 'releases', 'links'] | licență: http://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/
Pagina 1 | status 200
Pagina 2 | status 200
Pagina 3 | status 200
Pagina 4 | status 200

Release-uri: 500 | apariții de furnizori: 1480
Scheme principale: {'GB-COH': 728, 'LIPSĂ': 713, 'GB-NHS': 35, 'GB-CHC': 4}
Scheme adiționale: {}
Furnizori cu TVA: 0

Exemplu de furnizor (structura completă):
{
  "name": "Care UK Community Partnerships Ltd",
  "id": "GB-COH-2644862",
  "identifier": {
    "legalName": "Care UK Community Partnerships Ltd",
    "id": "2644862",
    "scheme": "GB-COH"
  },
  "address": {
    "streetAddress": "5 Churchill Place, 10th Floor,",
    "locality": "London",
    "region": "UKI42",
    "postalCode": "E14 5HU",
    "countryName": "United Kingdom"
  },
  "roles": [
    "supplier"
  ],
  "details": {
    "url": "https://www.care

In [5]:
import re
total, cu_ch, cu_url, cu_ambele, cu_cp = 0, 0, 0, 0, 0
scale = collections.Counter()
furn_unici = {}
url, cerere, pagini = BAZA, dict(params), 0
while url and pagini < 5:
    date = cere(url, cerere).json()
    for rel in date.get("releases", []):
        for p in rel.get("parties", []):
            if "supplier" not in p.get("roles", []):
                continue
            ident = p.get("identifier", {})
            cheie = ident.get("id") or p.get("name")
            furn_unici[cheie] = p
    nxt = date.get("links", {}).get("next")
    url, cerere = (nxt, None) if nxt else (None, None)
    pagini += 1
    time.sleep(1)

for p in furn_unici.values():
    total += 1
    ch = p.get("identifier", {}).get("scheme") == "GB-COH"
    u = bool(p.get("details", {}).get("url"))
    cu_ch += ch; cu_url += u; cu_ambele += (ch and u)
    cu_cp += bool(p.get("address", {}).get("postalCode"))
    scale[p.get("details", {}).get("scale", "lipsă")] += 1

print("Furnizori unici:", total)
print(f"Cu număr CH:      {cu_ch} ({cu_ch/total:.0%})")
print(f"Cu site (url):    {cu_url} ({cu_url/total:.0%})")
print(f"Cu CH și site:    {cu_ambele} ({cu_ambele/total:.0%})")
print(f"Cu cod poștal:    {cu_cp} ({cu_cp/total:.0%})")
print("Mărime (scale):", dict(scale))

Furnizori unici: 1371
Cu număr CH:      697 (51%)
Cu site (url):    518 (38%)
Cu CH și site:    436 (32%)
Cu cod poștal:    1129 (82%)
Mărime (scale): {'large': 408, 'sme': 959, 'lipsă': 4}


In [6]:
import random
from urllib.parse import urlparse
import pandas as pd

def norm_ch(x):
    x = str(x).strip().upper()
    if x.isdigit():
        return x.zfill(8)
    m = re.match(r"([A-Z]+)(\d+)$", x)
    return m.group(1) + m.group(2).zfill(8 - len(m.group(1))) if m else x

def domeniu(u):
    u = u if u.startswith("http") else "http://" + u
    d = urlparse(u).netloc.lower()
    return d[4:] if d.startswith("www.") else d

bazin = [p for p in furn_unici.values()
         if p.get("identifier", {}).get("scheme") == "GB-COH" and p.get("details", {}).get("url")]
print("Bazin (CH + site):", len(bazin))

rng = random.Random(42)
alesi = rng.sample(bazin, 30)

esantion = pd.DataFrame([{
    "company_number": norm_ch(p["identifier"]["id"]),
    "nume": p.get("identifier", {}).get("legalName") or p.get("name"),
    "url": p["details"]["url"],
    "domeniu": domeniu(p["details"]["url"]),
    "postcode": p.get("address", {}).get("postalCode", ""),
    "scale": p.get("details", {}).get("scale", ""),
} for p in alesi])

esantion.to_csv("../data/samples/fat_esantion_30.csv", index=False)
print(esantion[["company_number", "nume", "domeniu", "scale"]].to_string(index=False))
print("\nMărime în eșantion:", esantion.scale.value_counts().to_dict())

Bazin (CH + site): 436
company_number                                                      nume                            domeniu scale
      03553977                     Salisbury Management Services Limited         salisburyhousewirral.co.uk   sme
      14174577                                   Brimstage Manor Limited                  springcare.org.uk   sme
      01069092                                    Beyond Support Limited           beyondsupportlimited.com large
      10707210                                St Mary's Nursing Home Ltd             stmarysnursinghome.com   sme
      06962108                         Optima Health and Social Care Ltd           courtyardcaregroup.co.uk   sme
      12359267                                    Future Living Care Ltd             futurelivingcare.co.uk   sme
      13838058                                            SAS Care Group                   sascaregroup.com   sme
      10681098                            Looking Forward Activit

In [7]:
from collections import Counter
furn_per_release, cpv_furn = [], Counter()
url, cerere, pagini = BAZA, dict(params), 0
while url and pagini < 5:
    date = cere(url, cerere).json()
    for rel in date.get("releases", []):
        n = sum("supplier" in p.get("roles", []) for p in rel.get("parties", []))
        cpv = rel.get("tender", {}).get("classification", {}).get("id", "??")[:2]
        furn_per_release.append(n)
        cpv_furn[cpv] += n
    nxt = date.get("links", {}).get("next")
    url, cerere = (nxt, None) if nxt else (None, None)
    pagini += 1
    time.sleep(1)

s = pd.Series(furn_per_release)
print("Furnizori per contract: mediana", s.median(), "| max", s.max())
print("Contracte cu >10 furnizori:", (s > 10).sum(), "din", len(s),
      "| au", s[s > 10].sum(), "din", s.sum(), "apariții de furnizori")
print("\nApariții de furnizori pe diviziune CPV (primele 2 cifre):")
print(cpv_furn.most_common(8))

Furnizori per contract: mediana 1.0 | max 76
Contracte cu >10 furnizori: 26 din 500 | au 904 din 1480 apariții de furnizori

Apariții de furnizori pe diviziune CPV (primele 2 cifre):
[('85', 789), ('60', 198), ('34', 87), ('33', 76), ('80', 49), ('45', 45), ('48', 33), ('72', 33)]


In [8]:
import random
EXCLUDE_CPV = {"85"}        # pune set() dacă vrei să păstrezi și sectorul de îngrijire
N, SEED = 30, 42

candidati_pe_contract = []
url, cerere, pagini = BAZA, dict(params), 0
while url and pagini < 10:                     # 10 × 100 = 1.000 de contracte
    date = cere(url, cerere).json()
    for rel in date.get("releases", []):
        cpv = rel.get("tender", {}).get("classification", {}).get("id", "??")
        if cpv[:2] in EXCLUDE_CPV:
            continue
        ok = [p for p in rel.get("parties", [])
              if "supplier" in p.get("roles", [])
              and p.get("identifier", {}).get("scheme") == "GB-COH"
              and p.get("details", {}).get("url")]
        if ok:
            candidati_pe_contract.append((rel.get("ocid"), cpv, ok))
    nxt = date.get("links", {}).get("next")
    url, cerere = (nxt, None) if nxt else (None, None)
    pagini += 1
    time.sleep(1)

print("Contracte cu cel puțin un furnizor eligibil:", len(candidati_pe_contract))

rng = random.Random(SEED)
contracte = rng.sample(candidati_pe_contract, min(N, len(candidati_pe_contract)))
randuri, vazute = [], set()
for ocid, cpv, ok in contracte:
    p = rng.choice(ok)
    ch = norm_ch(p["identifier"]["id"])
    if ch in vazute:                           # același furnizor din alt contract
        continue
    vazute.add(ch)
    randuri.append({
        "company_number": ch,
        "nume": p.get("identifier", {}).get("legalName") or p.get("name"),
        "url": p["details"]["url"], "domeniu": domeniu(p["details"]["url"]),
        "postcode": p.get("address", {}).get("postalCode", ""),
        "scale": p.get("details", {}).get("scale", ""), "cpv": cpv, "ocid": ocid,
    })

esantion = pd.DataFrame(randuri)
esantion.to_csv("../data/samples/fat_esantion_30.csv", index=False)
print(esantion[["company_number", "nume", "domeniu", "scale", "cpv"]].to_string(index=False))
print("\nMărime:", esantion.scale.value_counts().to_dict())
print("Diviziuni CPV:", esantion.cpv.str[:2].value_counts().to_dict())

Contracte cu cel puțin un furnizor eligibil: 10
company_number                              nume            domeniu scale      cpv
      02877612                INFOSHARE+ LIMITED  infoshareplus.com   sme 72000000
      00968498 NEC Software Solutions UK Limited         necsws.com large 48000000
      00673817         TW ENGINEERING CO LIMITED        tweng.co.uk   sme 43830000
      10389519               True Compliance Ltd      camden.gov.uk   sme 72000000
      02040690             Bromcom Computers PLC        bromcom.com large 72268000
      OC308658                           TLT LLP            tlt.com large 79100000
      07538903                  MPS UK Group Ltd     liv-coll.ac.uk   sme 39100000
      04771037                  ACADEMIA LIMITED     academia.co.uk large 30200000
      03341304  Stockton on Tees Borough Council    stockton.gov.uk       72222300
      SC007906              Wrights Health Group wrightdental.co.uk   sme 33192410

Mărime: {'sme': 5, 'large': 4, '': 1}


In [9]:
import json, random, re, time
import pandas as pd

FISIER = "../data/registre/fat_award_2026_01_09.jsonl"
params_lung = {"updatedFrom": "2026-01-01T00:00:00", "updatedTo": "2026-10-01T00:00:00",
               "limit": 100, "stages": "award"}
MAX_PAGINI = 60

# 1. Descărcare (o singură dată; dacă fișierul există, îl citește de pe disc)
import os
if not os.path.exists(FISIER):
    url, cerere, pagini = BAZA, dict(params_lung), 0
    with open(FISIER, "w") as f:
        while url and pagini < MAX_PAGINI:
            r = cere(url, cerere)
            if r.status_code != 200:
                print("Oprit la pagina", pagini, "status", r.status_code); break
            date = r.json()
            for rel in date.get("releases", []):
                f.write(json.dumps(rel) + "\n")
            nxt = date.get("links", {}).get("next")
            url, cerere = (nxt, None) if nxt else (None, None)
            pagini += 1
            if pagini % 10 == 0:
                print("pagini:", pagini)
            time.sleep(1)
releases = [json.loads(l) for l in open(FISIER)]
print("Contracte descărcate:", len(releases))

# 2. Candidați: un furnizor eligibil per contract, fără CPV 85, fără site-uri publice
SITE_PUBLIC = re.compile(r"\.(gov|ac|nhs|sch|police|mod|parliament)\.uk$")
EXCLUDE_CPV = {"85"}

def seamana(nume, dom):
    cuvinte = [w for w in re.findall(r"[a-z]{4,}", nume.lower())
               if w not in {"limited", "group", "services", "holdings", "company", "solutions"}]
    return any(w in dom.replace("-", "") for w in cuvinte)

candidati, excluse = [], {"cpv85": 0, "site_public": 0}
for rel in releases:
    cpv = rel.get("tender", {}).get("classification", {}).get("id", "??")
    if cpv[:2] in EXCLUDE_CPV:
        excluse["cpv85"] += 1; continue
    ok = []
    for p in rel.get("parties", []):
        if ("supplier" in p.get("roles", []) and p.get("identifier", {}).get("scheme") == "GB-COH"
                and p.get("details", {}).get("url")):
            if SITE_PUBLIC.search(domeniu(p["details"]["url"])):
                excluse["site_public"] += 1; continue
            ok.append(p)
    if ok:
        candidati.append((rel.get("ocid"), cpv, ok))
print("Contracte eligibile:", len(candidati), "| excluse:", excluse)

# 3. Eșantion: contract aleator → furnizor aleator, fără dubluri
rng = random.Random(42)
rng.shuffle(candidati)
randuri, vazute = [], set()
for ocid, cpv, ok in candidati:
    p = rng.choice(ok)
    ch = norm_ch(p["identifier"]["id"])
    if ch in vazute:
        continue
    vazute.add(ch)
    nume = p.get("identifier", {}).get("legalName") or p.get("name")
    dom = domeniu(p["details"]["url"])
    randuri.append({"company_number": ch, "nume": nume, "url": p["details"]["url"], "domeniu": dom,
                    "postcode": p.get("address", {}).get("postalCode", ""),
                    "scale": p.get("details", {}).get("scale", ""), "cpv": cpv, "ocid": ocid,
                    "domeniu_seamana_cu_numele": seamana(nume, dom), "domeniu_verificat_manual": ""})
    if len(randuri) == 30:
        break

esantion = pd.DataFrame(randuri)
esantion.to_csv("../data/samples/fat_esantion_30_v3.csv", index=False)
print(esantion[["company_number", "nume", "domeniu", "scale", "domeniu_seamana_cu_numele"]].to_string(index=False))
print("\nMărime:", esantion.scale.value_counts().to_dict())
print("Diviziuni CPV:", esantion.cpv.str[:2].value_counts().to_dict())
print("Domeniu care nu seamănă cu numele:", (~esantion.domeniu_seamana_cu_numele).sum(), "din", len(esantion))

pagini: 10
  status 429, aștept 120s (Retry-After)
pagini: 20
pagini: 30
  status 429, aștept 120s (Retry-After)
pagini: 40
pagini: 50
  status 429, aștept 120s (Retry-After)
pagini: 60
Contracte descărcate: 6000
Contracte eligibile: 48 | excluse: {'cpv85': 2174, 'site_public': 8}
company_number                                                 nume                           domeniu scale  domeniu_seamana_cu_numele
      08668845                                 Inspectas Compliance                   inspectas.co.uk   sme                       True
      13524377                                 Vilolet-Shift UK Ltd                 violetshift.co.uk   sme                       True
      16605182                                            AIION Ltd                       aiion.co.uk   sme                       True
      05199107                      People Asset Management Limited                    pamgroup.co.uk   sme                      False
      NI038469                             

In [10]:
import duckdb, re
import pandas as pd

CH_CSV = "../data/BasicCompanyDataAsOneFile-2026-09-01.csv"
es = pd.read_csv("../data/samples/fat_esantion_30_v3.csv", dtype=str, keep_default_na=False)

def norm_nume(s):
    s = s.upper().replace("&", "AND")
    s = re.sub(r"\bLIMITED\b", "LTD", s)
    s = re.sub(r"\(THE\)|\bTHE\b", "", s)
    return re.sub(r"[^A-Z0-9]", "", s)

con = duckdb.connect()
con.register("es", es[["company_number", "nume", "domeniu", "postcode"]])
rez = con.execute(f"""
    SELECT es.company_number, es.nume AS nume_fat, ch.companyname AS nume_ch,
           ch.companystatus, ch.accountsaccountcategory, es.domeniu,
           es.postcode AS cp_fat, ch.regaddresspostcode AS cp_ch
    FROM es LEFT JOIN read_csv('{CH_CSV}', normalize_names=true, ignore_errors=true, all_varchar=true) ch
      ON ch.companynumber = es.company_number
""").df()

rez["nume_egal"] = [isinstance(b, str) and norm_nume(a) == norm_nume(b) for a, b in zip(rez.nume_fat, rez.nume_ch)]
cp = lambda s: re.sub(r"\s", "", str(s).upper()) if isinstance(s, str) else ""
rez["cp_egal"] = [cp(a) != "" and cp(a) == cp(b) for a, b in zip(rez.cp_fat, rez.cp_ch)]

pd.set_option("display.width", 250)
print("Găsite în CH:", rez.nume_ch.notna().sum(), "din", len(rez))
print("Nume identic:", rez.nume_egal.sum(), "| cod poștal identic:", rez.cp_egal.sum())
print("\nDe verificat (nume diferit sau negăsit în CH):")
print(rez.loc[~rez.nume_egal, ["company_number", "nume_fat", "nume_ch", "domeniu"]].to_string(index=False))
print("\nCategorii de conturi:", rez.accountsaccountcategory.value_counts().to_dict())
rez.to_csv("../data/results/fat_esantion_30_v3_verificat_ch.csv", index=False)

Găsite în CH: 30 din 30
Nume identic: 24 | cod poștal identic: 22

De verificat (nume diferit sau negăsit în CH):
company_number                          nume_fat                          nume_ch                           domeniu
      03460378                       Bouygues UK          BOUYGUES (U.K.) LIMITED                   bouygues-uk.com
      08668845              Inspectas Compliance         INSPECTAS COMPLIANCE LTD                   inspectas.co.uk
      06711962                           ProofID                  PROOFID LIMITED                       proofid.com
      00673817         TW ENGINEERING CO LIMITED T.W. ENGINEERING COMPANY LIMITED                       tweng.co.uk
      13524377              Vilolet-Shift UK Ltd          VIOLET-SHIFT UK LIMITED                 violetshift.co.uk
      00387371 Kind & Company (Builders) Limited    KIND & CO. (BUILDERS) LIMITED londonconstructionprogramme.co.uk

Categorii de conturi: {'FULL': 13, 'TOTAL EXEMPTION FULL': 8, 'GROUP': 5,

In [11]:
verificare = {
    "08668845": "da", "13524377": "da", "16605182": "da", "05199107": "grup",
    "NI038469": "da", "06711962": "da", "00557725": "grup", "00191596": "da",
    "07291125": "da", "00673817": "da", "01616977": "neverificat", "OC308658": "da",
    "06832618": "da", "03341304": "neverificat", "06296993": "da", "08549843": "da",
    "08363432": "neverificat", "OC304417": "da", "01383511": "neverificat", "06667367": "da",
    "03460378": "da", "10398586": "da", "03789359": "da", "16314547": "da",
    "OC307212": "da", "RC000648": "nu", "00387371": "nu", "03146250": "da",
    "00968498": "neverificat", "04771037": "da",
}
es = pd.read_csv("../data/samples/fat_esantion_30_v3.csv", dtype=str, keep_default_na=False)
es["domeniu_verificat_manual"] = es.company_number.map(verificare)
es.to_csv("../data/samples/fat_esantion_30_v3.csv", index=False)
print(es.domeniu_verificat_manual.value_counts())

domeniu_verificat_manual
da             21
neverificat     5
grup            2
nu              2
Name: count, dtype: int64
